# Bootstrap CI for Financial Time Series

Standard bootstrap assumes iid data — catastrophically wrong for financial returns
which exhibit autocorrelation, volatility clustering, and fat tails.

This notebook shows the correct approach using block bootstrap methods.


In [1]:
import numpy as np
import bootstrapx
from bootstrapx import bootstrap

# Simulate AR(1)-GARCH-like returns
rng = np.random.default_rng(42)
n = 1000
returns = np.zeros(n)
vol = np.ones(n)
for t in range(1, n):
    vol[t] = np.sqrt(0.01 + 0.1*returns[t-1]**2 + 0.85*vol[t-1]**2)
    returns[t] = 0.05*returns[t-1] + vol[t]*rng.standard_normal()
print(f"Sharpe (annualized): {returns.mean()/returns.std()*np.sqrt(252):.3f}")

Sharpe (annualized): -0.458


In [2]:
def sharpe(r):
    return r.mean() / r.std() * np.sqrt(252)

# IID bootstrap (WRONG: ignores serial dependence)
r_iid = bootstrap(returns, sharpe, method="bca", n_resamples=4999, random_state=0)

# Moving Block Bootstrap (preserves autocorrelation structure)
r_mbb = bootstrap(returns, sharpe, method="mbb", block_length=20, n_resamples=4999, random_state=0)

# Stationary Bootstrap (Politis & Romano 1994)
r_stat = bootstrap(returns, sharpe, method="stationary", mean_block=20.0, n_resamples=4999, random_state=0)

for name, r in [("IID BCa", r_iid), ("MBB", r_mbb), ("Stationary", r_stat)]:
    ci = r.confidence_interval
    print(f"{name:12s}: SE={r.standard_error:.4f}  CI=[{ci.low:.4f}, {ci.high:.4f}]")

print()
print("MBB/Stationary SE is larger = correct for autocorrelated data")

IID BCa     : SE=0.5022  CI=[-1.4496, 0.5094]
MBB         : SE=0.5024  CI=[-1.4782, 0.5047]
Stationary  : SE=0.4710  CI=[-1.4017, 0.4237]

MBB/Stationary SE is larger = correct for autocorrelated data
